
## Pentru testare vom folosi 2 clase:
- RAGSecurityAgent - pentru analiza
- SelfImprovingFeedbackAgent - pentru stocare

In [1]:
import sys
import os
project_root = os.path.abspath(os.path.join(os.path.dirname(os.path.abspath("demo_pipeline.ipynb")), ".."))
os.chdir(project_root)
sys.path.insert(0, project_root)
print(os.getcwd())
#asta ca sa mearga pt toata lumea automat... si ca gaseasca vectorstore
# DACA NU MERGE LA PRIMUL RUN, STOP KERNEL AND RESTART

from src.agents.security_agent import RAGSecurityAgent
from src.agents.feedback_agent import SelfImprovingFeedbackAgent
from src.dtos import CodeFileDTO, Language, FeedbackDTO
from langchain_openai import OpenAIEmbeddings
from src.tools.vector_tools import load_corpus, build_index
import chromadb
security_agent = RAGSecurityAgent()
feedback_agent = SelfImprovingFeedbackAgent()
feedback_agent.reset_memory()


C:\Facultate\An_3\Semestrul_2\SI\Lab\ProiectAAI\proiect-aai-barnachirautan


In [2]:
cod_vulnerabil = """
def get_user(uid):
    query = f"SELECT * FROM users WHERE id={uid}"
    cursor.execute(query)
"""

file_dto = CodeFileDTO(
    file_path="test/test_db.py",
    language=Language.PYTHON,
    content=cod_vulnerabil,
    lines_of_code=4,
    functions=[],
    imports=[],
    dependencies=[]
)

security_agent.threshold = 0.1
vulns = security_agent.scan(file_dto)
for v in vulns:
    print(f"[{v.severity}] {v.title} — {v.description[:100]}")

[VulnerabilitySeverity.CRITIC] SQL Injection Vulnerability — The use of Python f-string interpolation to construct SQL queries directly from user input allows fo


In [3]:
from datetime import datetime

if vulns:
    feedback = FeedbackDTO(
        feedback_id="FB-DEMO-001",
        original_finding_id=vulns[0].id,
        file_path="test/test_db.py",
        code_snippet=cod_vulnerabil,
        agent_verdict=vulns[0].title,
        is_false_positive=True,
        human_comment="Cod de test, uid e fixture fix, nu input utilizator real",
        timestamp=datetime.utcnow().isoformat()
    )

    feedback_agent.store_feedback(feedback)
    print(f"Feedback stocat: {feedback.feedback_id}")
    print(f"Verdict: FP — {feedback.human_comment}")
else:
    print("Nu s-au gasit vulnerabilitati — incearca cu threshold mai mic")

C:\Users\Razvan\AppData\Local\Temp\ipykernel_15776\2607539242.py:12: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp=datetime.utcnow().isoformat()


Feedback stocat: FB-DEMO-001
Verdict: FP — Cod de test, uid e fixture fix, nu input utilizator real


In [4]:

# reseteaza si restocheaza in aceeasi sesiune
feedback_agent.reset_memory()
feedback_agent.store_feedback(feedback)

cod_similar = """
def get_product(pid):
    query = f"SELECT * FROM products WHERE id={pid}"
    cursor.execute(query)
"""

file_dto2 = CodeFileDTO(
    file_path="test/test_products.py",
    language=Language.PYTHON,
    content=cod_similar,
    lines_of_code=4,
    functions=[],
    imports=[],
    dependencies=[]
)

# verifica distanta
from langchain_openai import OpenAIEmbeddings
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
client_mem = chromadb.PersistentClient(path="memory/feedback_index/")
col = client_mem.get_collection("episodic_memory")
vector = embeddings.embed_query("SQL query f-string user input cursor execute")
docs = col.query(query_embeddings=[vector], n_results=1)
print(f"Distanta: {docs['distances']}")
score = 1 - docs['distances'][0][0]
print(f"Scor similaritate: {score:.3f}")

# recupereaza nota
feedback_agent.memory_threshold = 0.1 # TH MIC CA AM DOAR UN FEEDBACK IN MEMORIE
nota = feedback_agent.augment_context("SQL query f-string user input cursor execute")
print(f"Nota din memorie:\n{nota if nota else 'Nicio nota gasita'}\n")

# re-analiza
vulns2 = security_agent.scan(file_dto2)
print("\nVulnerabilitati dupa feedback:")
for v in vulns2:
    print(f"[{v.severity}] {v.title}")

Distanta: [[0.5480747222900391]]
Scor similaritate: 0.452
Nota din memorie:
Nota din review-urile anterioare: [FP] SQL Injection Vulnerability in test/test_db.py: Cod de test, uid e fixture fix, nu input utilizator real. Cod: 
def get_user(uid):
    query = f"SELECT * FROM users WHERE id={uid}"
    cursor.execute(query)


Vulnerabilitati dupa feedback:
[VulnerabilitySeverity.CRITIC] SQL Injection


Agentu a gasit in memorie un feedback similar, ca si FP . Threshold u e mic ca in test exista doar un feedback in fisieru json, odata cu cresterea numarului de feedback uri , va creste si threshold u. Acest vulnerabilityDTO e trimis la security agent mai departe si va decide ce sa faca cu el, unde are toate datele despre vulnerabilitate, cod si tipu, cele mai importante, dandu-i context.